# Nuovo noteboook rispetto al lavoro di Claudia,semplicemente per calcolare le matrici colorate per individuare la miglior combinazione layer-head

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
model_type = "fine_grained"
#model_type = "full_fine" #Commenta in base al modello che devi usare

base = f"/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/{model_type}"
output_dir = f"/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/{model_type}/matrici"
os.makedirs(output_dir, exist_ok=True)

layers = list(range(-1, -13, -1))  #i layer hanno indice negativo da [-1],l'ultimo, a [-12],il primo

In [ ]:
dfs = []
for layer in layers:
    path = os.path.join(base, f"metrics_heads[{layer}].csv")
    if not os.path.exists(path):  #solo un controllo per essere sicuri che ci siano tutti i file
        print(f" File mancante: {path} ")
        continue
    df = pd.read_csv(path)
    df['layer'] = layer
    dfs.append(df)

df_all = pd.concat(dfs).reset_index(drop=True)
print(f"Caricati {df_all['layer'].nunique()} layer, {len(df_all)} righe totali")


In [ ]:
for metric, cmap in [('NSS', 'YlGnBu'), ('EMD', 'YlOrRd_r')]:

    matrix = df_all.pivot_table(index='layer', columns='head',
                                values=metric, aggfunc='mean')
    matrix = matrix.reindex(index=layers)

    # Sistemo etichette
    ylabels = [f"{13 - abs(l)}°" for l in matrix.index]   # "(-1)"= 12°
    xlabels = [str(h + 1) for h in matrix.columns]        # head 1-14

    plt.figure(figsize=(16, 8))
    sns.heatmap(matrix, annot=True, fmt=".2f",
                cmap=cmap, linewidths=0.5, linecolor='white',
                annot_kws={"fontsize": 8},
                xticklabels=xlabels, yticklabels=ylabels,
                cbar_kws={'label': f'{metric} medio'})

    plt.title(f"{metric} medio Layer × Head — mCLIP ({model_type})",
              fontsize=16, weight='bold')
    plt.xlabel("Head", fontsize=13)
    plt.ylabel("Layer", fontsize=13)
    plt.yticks(rotation=0)      # per avere le etichette dei layer in orizzontale e non in verticale
    plt.tight_layout()

    filename = os.path.join(output_dir, f"matrix_{metric}_{model_type}.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Salvato: {filename}")

    if metric == 'NSS':
        best = matrix.stack().idxmax()
        val  = matrix.stack().max()
    else:
        best = matrix.stack().idxmin()
        val  = matrix.stack().min()
    print(f"  Migliore {metric}: layer {best[0]}, head {best[1]} Valore: {val:.3f}")

### VERIFICA MEME

In [ ]:

#Ottengo il migliore e peggiore meme per entrambe le metriche
model_type = "fine_grained"
#model_type = "full_fine" #Commenta in base al modello che devi utilizzare


base = f"/content/gdrive/MyDrive/LM-Info-Pizzo-Davide/output/head_attention/{model_type}"
csv_path = f"{base}/metrics_heads[-9].csv"

HEAD_IDX = 3   # 4ª head

df = pd.read_csv(csv_path)
df_h = df[df['head'] == HEAD_IDX]
print(f"Modello: {model_type}  |  righe (meme) per head {HEAD_IDX+1}: {len(df_h)}\n")

best_nss  = df_h.loc[df_h['NSS'].idxmax()]
worst_nss = df_h.loc[df_h['NSS'].idxmin()]
print("NSS")
print(f"  Migliore: {best_nss['meme']}  →  NSS = {best_nss['NSS']:.4f}")
print(f"  Peggiore: {worst_nss['meme']}  →  NSS = {worst_nss['NSS']:.4f}\n")


best_emd  = df_h.loc[df_h['EMD'].idxmin()]
worst_emd = df_h.loc[df_h['EMD'].idxmax()]
print("EMD")
print(f"  Migliore: {best_emd['meme']}  →  EMD = {best_emd['EMD']:.4f}")
print(f"  Peggiore: {worst_emd['meme']}  →  EMD = {worst_emd['EMD']:.4f}")

 ### STESSE MATRICI MA CON LA MEDIANA

In [ ]:
for metric, cmap in [('NSS', 'YlGnBu'), ('EMD', 'YlOrRd_r')]:

    matrix = df_all.pivot_table(index='layer', columns='head',
                                values=metric, aggfunc='median')   # MEDIANA
    matrix = matrix.reindex(index=layers)

    # Sistemo etichette
    ylabels = [f"{13 - abs(l)}°" for l in matrix.index]
    xlabels = [str(h + 1) for h in matrix.columns]

    plt.figure(figsize=(16, 8))
    sns.heatmap(matrix, annot=True, fmt=".2f",
                cmap=cmap, linewidths=0.5, linecolor='white',
                annot_kws={"fontsize": 8},
                xticklabels=xlabels, yticklabels=ylabels,
                cbar_kws={'label': f'{metric} mediano'})

    plt.title(f"{metric} mediano Layer × Head — mCLIP ({model_type})",
              fontsize=16, weight='bold')
    plt.xlabel("Head", fontsize=13)
    plt.ylabel("Layer", fontsize=13)
    plt.yticks(rotation=0)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"matrix_{metric}_median_{model_type}.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Salvato: {filename}")

    if metric == 'NSS':
        best = matrix.stack().idxmax()
        val  = matrix.stack().max()
    else:
        best = matrix.stack().idxmin()
        val  = matrix.stack().min()
    print(f"  Migliore {metric} (mediana): layer {best[0]}, head {best[1]} → {val:.3f}")